<a href="https://colab.research.google.com/github/beckysianga/wileyproject.github.io/blob/main/FL_AS_HDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required packages
!pip -q install gradio pandas numpy scikit-learn matplotlib torch scipy openpyxl


import os, math, copy, random, tempfile, zipfile, json
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ---------------------------
# Data cleaning and labels
# ---------------------------
def safe_to_numeric_series(s: pd.Series) -> pd.Series:
    try:
        return pd.to_numeric(s)
    except Exception:
        return s

def infer_target_column(df: pd.DataFrame, preferred: Optional[str] = None) -> str:
    if preferred and preferred in df.columns:
        return preferred
    common = ["target", "label", "class", "Class", "diagnosis", "Diagnosis",
              "Outcome", "outcome", "Diabetes_binary", "Grade", "grade"]
    for c in common:
        if c in df.columns:
            return c
    small = [c for c in df.columns if df[c].nunique(dropna=True) <= 10]
    return small[-1] if small else df.columns[-1]

def make_binary_labels(y: pd.Series, positive_label: str = "") -> np.ndarray:
    s = y.copy()
    if positive_label:
        pos = str(positive_label).strip().lower()
        return (s.astype(str).str.strip().str.lower() == pos).astype(np.int64).values

    # WDBC diagnosis mapping
    st = s.astype(str).str.strip().str.lower()
    if set(st.dropna().unique()).issubset({"m", "b"}):
        return st.map({"m": 1, "b": 0}).astype(np.int64).values

    # Common glioma label mapping
    high_keys = ["hgg", "high", "gbm", "glioblastoma", "grade iv", "grade 4", "g4", "iii", "iv"]
    low_keys = ["lgg", "low", "grade ii", "grade 2", "grade i", "grade 1", "g1", "g2"]
    mapped = []
    has_keyword = False
    for v in st.values:
        if any(k in v for k in high_keys):
            mapped.append(1); has_keyword = True
        elif any(k in v for k in low_keys):
            mapped.append(0); has_keyword = True
        else:
            mapped.append(np.nan)
    if has_keyword and not pd.isna(mapped).any():
        return np.array(mapped, dtype=np.int64)

    # Numeric rules
    try:
        sn = pd.to_numeric(s)
        uniq = sorted(pd.unique(sn.dropna()))
        if len(uniq) == 2:
            return (sn.values == uniq[-1]).astype(np.int64)
        # grade-like rule: >= 3 is high risk/high grade
        if len(uniq) > 2 and len(uniq) <= 10:
            return (sn.astype(float).values >= 3.0).astype(np.int64)
        return (sn.astype(float).values > np.nanmedian(sn.astype(float).values)).astype(np.int64)
    except Exception:
        pass

    # Final fallback: factorize exactly two classes
    codes, uniques = pd.factorize(st)
    if len(uniques) != 2:
        raise ValueError(f"Target column must be binary or convertible to binary. Found classes: {list(uniques[:10])}")
    return codes.astype(np.int64)

def clean_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(axis=1, how="all").copy()
    df = df.replace(["?", "NA", "N/A", "na", "n/a", "null", "None", ""], np.nan)
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = safe_to_numeric_series(df[c])
    for c in df.columns:
        if df[c].dtype == object:
            df[c], _ = pd.factorize(df[c].astype(str))
    for c in df.columns:
        if df[c].isna().any():
            med = df[c].median() if pd.api.types.is_numeric_dtype(df[c]) else 0
            df[c] = df[c].fillna(med)
    return df

def load_uploaded_csv(file_obj, target_column="", positive_label="", test_size=0.20, val_size=0.10, seed=42):
    path = file_obj.name if hasattr(file_obj, "name") else file_obj
    df_raw = pd.read_csv(path)
    df_raw = df_raw.dropna(axis=0, how="all").copy()
    target = infer_target_column(df_raw, target_column.strip() or None)
    y = make_binary_labels(df_raw[target], positive_label)
    Xdf = df_raw.drop(columns=[target])
    # Drop obvious ID columns unless they are the only feature
    id_like = [c for c in Xdf.columns if c.lower() in {"id", "patient_id", "case_id", "subject", "sample", "name"}]
    if len(Xdf.columns) - len(id_like) >= 1:
        Xdf = Xdf.drop(columns=id_like)
    Xdf = clean_features(Xdf)
    X = Xdf.values.astype(np.float32)
    if len(np.unique(y)) < 2:
        raise ValueError("The target column has only one class after binary conversion.")
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )
    rel_test = 0.50
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=rel_test, stratify=y_tmp, random_state=seed
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    metadata = {
        "rows": int(len(df_raw)),
        "features": int(X.shape[1]),
        "target": target,
        "positive_rate": float(np.mean(y)),
        "feature_names": list(Xdf.columns)
    }
    return X_train, X_val, X_test, y_train, y_val, y_test, scaler, metadata, Xdf.columns.tolist()

# ---------------------------
# Model
# ---------------------------
class TabDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class FCNN(nn.Module):
    def __init__(self, d_in: int, hidden=(128, 64), dropout=0.08):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def bce_logits_loss(logits, y, pos_weight=None):
    return nn.functional.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

@torch.no_grad()
def predict_proba(model, X, batch_size=2048):
    model.eval()
    dl = DataLoader(TabDataset(X, np.zeros(len(X))), batch_size=batch_size, shuffle=False)
    out = []
    for xb, _ in dl:
        xb = xb.to(DEVICE)
        out.append(torch.sigmoid(model(xb)).cpu().numpy().ravel())
    return np.concatenate(out) if out else np.array([])

def eval_metrics(model, X, y, batch_size=2048):
    prob = predict_proba(model, X, batch_size)
    pred = (prob >= 0.5).astype(int)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "auprc": float(average_precision_score(y, prob)),
    }
    try:
        out["auroc"] = float(roc_auc_score(y, prob))
    except Exception:
        out["auroc"] = np.nan
    return out

# ---------------------------
# Federated learning utilities
# ---------------------------
def make_non_iid_partitions(X, y, n_clients=8, dirichlet=0.6, seed=42):
    rng = np.random.default_rng(seed)
    client_parts = [[] for _ in range(n_clients)]
    for cls in np.unique(y):
        idxs = np.where(y == cls)[0]
        rng.shuffle(idxs)
        p = rng.dirichlet([dirichlet] * n_clients)
        cuts = (p * len(idxs)).astype(int)
        cuts[-1] += len(idxs) - cuts.sum()
        s = 0
        for i, c in enumerate(cuts):
            client_parts[i].extend(idxs[s:s+c])
            s += c
    clients = []
    for part in client_parts:
        arr = np.array(part, dtype=int)
        rng.shuffle(arr)
        clients.append(arr)
    return clients

def l2_norm_update(update):
    return float(math.sqrt(sum(float(torch.sum(t.detach() ** 2).cpu()) for t in update) + 1e-12))

def clip_update(update, clip_norm):
    n = l2_norm_update(update)
    if n <= clip_norm:
        return update, n
    sc = clip_norm / (n + 1e-12)
    return [t * sc for t in update], n

def add_laplace_noise(update, scale_b):
    if scale_b <= 0:
        return update
    return [t + torch.distributions.Laplace(0.0, scale_b).sample(t.shape).to(t.device) for t in update]

def add_gaussian_noise(update, sigma):
    if sigma <= 0:
        return update
    return [t + sigma * torch.randn_like(t) for t in update]

def compute_feature_sensitivity(Xc, alpha=0.65, beta=0.35, eps=1e-12):
    if len(Xc) == 0:
        return 1.0
    std = Xc.std(axis=0)
    rng = Xc.max(axis=0) - Xc.min(axis=0)
    return float(np.mean(alpha * (std + eps) + beta * (rng + eps)))

def squash_positive(x, cap=3.0, eps=1e-12):
    x = max(float(x), eps)
    return float(cap * (x / (x + cap)))

def server_sensitivity_proxy(client_sens_list, clipped_norms, clip_bound, cap=3.0):
    s_med = float(np.median(np.array(client_sens_list))) if client_sens_list else 1.0
    medn = float(np.median(np.array(clipped_norms))) if clipped_norms else clip_bound
    closeness = float(np.clip(medn / (clip_bound + 1e-12), 0.0, 1.0))
    return squash_positive(s_med * (0.6 + 0.4 * closeness), cap=cap)

@dataclass
class ClipAdapt:
    init: float = 1.6
    q: float = 0.80
    ema: float = 0.95
    cmin: float = 0.25
    cmax: float = 6.0
    max_growth: float = 1.15
    min_growth: float = 0.90

def update_clip_bound(prev, raw_norms, cfg: ClipAdapt):
    if not raw_norms:
        return prev
    target = float(np.quantile(np.array(raw_norms, dtype=np.float64), cfg.q))
    newC = cfg.ema * prev + (1.0 - cfg.ema) * target
    newC = min(newC, prev * cfg.max_growth)
    newC = max(newC, prev * cfg.min_growth)
    return float(np.clip(newC, cfg.cmin, cfg.cmax))

class AdaptiveBudgetScheduler:
    def __init__(self, eps_total, rounds, warmup_rounds=12, warmup_mult=2.4,
                 late_gamma=1.2, stagnation_beta=1.6, min_eps=0.15, max_eps=2.5):
        self.eps_total = float(eps_total)
        self.rounds = int(rounds)
        self.warmup_rounds = int(warmup_rounds)
        self.warmup_mult = float(warmup_mult)
        self.late_gamma = float(late_gamma)
        self.stagnation_beta = float(stagnation_beta)
        self.min_eps = float(min_eps)
        self.max_eps = float(max_eps)
        self.used = 0.0

    def alloc(self, r, val_improve=0.0):
        remaining_rounds = self.rounds - r + 1
        remaining = max(self.eps_total - self.used, 0.0)
        if remaining_rounds <= 1:
            eps = remaining
        else:
            base = remaining / remaining_rounds
            mult = self.warmup_mult if r <= self.warmup_rounds else self.late_gamma
            if val_improve <= 1e-4:
                mult *= self.stagnation_beta
            eps = np.clip(base * mult, self.min_eps, self.max_eps)
            eps = min(float(eps), remaining)
        self.used += eps
        return float(eps)

def gaussian_sigma_for_eps_delta(eps, delta, sensitivity):
    eps = max(float(eps), 1e-12)
    delta = max(float(delta), 1e-12)
    return float(sensitivity * math.sqrt(2.0 * math.log(1.25 / delta)) / eps)

def weighted_average_updates(updates, weights=None):
    if weights is None:
        weights = np.ones(len(updates)) / len(updates)
    avg = []
    for params in zip(*updates):
        avg.append(sum(float(w) * p for w, p in zip(weights, params)))
    return avg

def apply_update(model, update, lr_global=1.0):
    with torch.no_grad():
        for p, du in zip(model.parameters(), update):
            p.add_(lr_global * du)

def local_train_client(global_model, Xc, yc, cfg, pos_weight=None):
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    dl = DataLoader(TabDataset(Xc, yc), batch_size=cfg.batch_size, shuffle=True)
    model.train()
    for _ in range(cfg.local_epochs):
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = bce_logits_loss(model(xb), yb, pos_weight=pos_weight)
            loss.backward()
            opt.step()
    update = []
    with torch.no_grad():
        for pn, po in zip(model.parameters(), global_model.parameters()):
            update.append((pn.data - po.data).detach().clone())
    m = eval_metrics(model, Xc, yc, batch_size=cfg.batch_size) if len(yc) > 0 else {"accuracy": 0}
    loss_proxy = 1.0 - m["accuracy"]
    return update, loss_proxy

@dataclass
class FLConfig:
    n_clients: int = 8
    rounds: int = 30
    local_epochs: int = 3
    batch_size: int = 256
    lr: float = 9e-4
    weight_decay: float = 1e-4
    fixed_clip_norm: float = 1.0
    clip_adapt: ClipAdapt = field(default_factory=ClipAdapt)
    alpha: float = 0.65
    beta: float = 0.35
    sens_cap: float = 3.0
    eps_ldp_total: float = 12.0
    eps_cdp_total: float = 8.0
    delta_cdp: float = 1e-5
    warmup_rounds: int = 12
    warmup_mult_ldp: float = 2.4
    warmup_mult_cdp: float = 3.0
    late_gamma_ldp: float = 1.20
    late_gamma_cdp: float = 1.15
    stagnation_beta: float = 1.6
    min_eps_round: float = 0.15
    max_eps_round: float = 2.5
    use_pos_weight: bool = True

@dataclass
class AblationFlags:
    adapt_clip: bool = True
    adapt_budget: bool = True
    adapt_client_sens: bool = True
    adapt_server_sens: bool = True

def run_fl_mode(mode, Xtr, ytr, Xva, yva, Xte, yte, d_in, cfg: FLConfig, flags: Optional[AblationFlags], seed=42):
    assert mode in {"NO_DP", "FIXED_HDP", "AS"}
    if flags is None:
        flags = AblationFlags()
    set_seed(seed)
    global_model = FCNN(d_in).to(DEVICE)
    client_idxs = make_non_iid_partitions(Xtr, ytr, cfg.n_clients, dirichlet=0.6, seed=seed)

    if cfg.use_pos_weight:
        pos = float(np.sum(ytr == 1)); neg = float(np.sum(ytr == 0))
        pw = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=DEVICE)
    else:
        pw = None

    eps_ldp_fixed = cfg.eps_ldp_total / cfg.rounds if cfg.eps_ldp_total > 0 else 0.0
    eps_cdp_fixed = cfg.eps_cdp_total / cfg.rounds if cfg.eps_cdp_total > 0 else 0.0

    sched_ldp = sched_cdp = None
    if mode == "AS" and flags.adapt_budget:
        sched_ldp = AdaptiveBudgetScheduler(cfg.eps_ldp_total, cfg.rounds, cfg.warmup_rounds,
                                            cfg.warmup_mult_ldp, cfg.late_gamma_ldp,
                                            cfg.stagnation_beta, cfg.min_eps_round, cfg.max_eps_round)
        sched_cdp = AdaptiveBudgetScheduler(cfg.eps_cdp_total, cfg.rounds, cfg.warmup_rounds,
                                            cfg.warmup_mult_cdp, cfg.late_gamma_cdp,
                                            cfg.stagnation_beta, cfg.min_eps_round, cfg.max_eps_round)

    fixed_sens = None
    if mode == "FIXED_HDP":
        fixed_sens = squash_positive(compute_feature_sensitivity(Xtr, cfg.alpha, cfg.beta), cfg.sens_cap)

    C_r = cfg.clip_adapt.init
    hist = {"round": [], "val_auprc": [], "clip_bound": [], "epsL_r": [], "epsC_r": [],
            "lap_b_med": [], "sigma": []}
    best_state, best_val = None, -1.0
    prev_val = None
    epsL_used, epsC_used = 0.0, 0.0

    for r in range(1, cfg.rounds + 1):
        val_improve = 0.001 if prev_val is None else float(hist["val_auprc"][-1] - prev_val)
        if mode == "NO_DP":
            epsL_r = epsC_r = 0.0
        elif mode == "FIXED_HDP":
            epsL_r, epsC_r = float(eps_ldp_fixed), float(eps_cdp_fixed)
        else:
            if flags.adapt_budget:
                epsL_r = sched_ldp.alloc(r, val_improve)
                epsC_r = sched_cdp.alloc(r, val_improve)
            else:
                epsL_r, epsC_r = float(eps_ldp_fixed), float(eps_cdp_fixed)
        epsL_used += epsL_r; epsC_used += epsC_r

        updates, raw_norms, clipped_norms, client_sens_list, losses = [], [], [], [], []
        for idx in client_idxs:
            if len(idx) == 0:
                continue
            Xc, yc = Xtr[idx], ytr[idx]
            update, loss_proxy = local_train_client(global_model, Xc, yc, cfg, pw)
            loss_proxy = max(loss_proxy, 1e-6)
            if mode == "NO_DP":
                up2 = update
                raw_n = l2_norm_update(up2)
                clip_n = raw_n
            else:
                clip_bound = C_r if (mode == "AS" and flags.adapt_clip) else cfg.fixed_clip_norm
                up2, raw_n = clip_update(update, clip_bound)
                clip_n = l2_norm_update(up2)

                if mode == "FIXED_HDP":
                    sens = fixed_sens
                else:
                    sens_raw = compute_feature_sensitivity(Xc, cfg.alpha, cfg.beta)
                    sens = squash_positive(sens_raw, cfg.sens_cap) if flags.adapt_client_sens else 1.0
                client_sens_list.append(sens)
                lap_b = sens / max(epsL_r, 1e-12) if epsL_r > 0 else 0.0
                up2 = add_laplace_noise(up2, lap_b)

            updates.append(up2); losses.append(loss_proxy)
            raw_norms.append(raw_n); clipped_norms.append(clip_n)

        if not updates:
            raise ValueError("No client received data. Reduce number of clients or check dataset size.")

        # loss-aware weights
        inv = 1.0 / (np.array(losses) + 1e-8)
        weights = inv / inv.sum()
        avg_update = weighted_average_updates(updates, weights)

        sigma = 0.0
        if mode != "NO_DP":
            if mode == "FIXED_HDP":
                s_server = fixed_sens
            else:
                s_server = server_sensitivity_proxy(client_sens_list, clipped_norms, C_r, cfg.sens_cap) if flags.adapt_server_sens else 1.0
            sigma = gaussian_sigma_for_eps_delta(epsC_r, cfg.delta_cdp, s_server) if epsC_r > 0 else 0.0
            avg_update = add_gaussian_noise(avg_update, sigma)

        apply_update(global_model, avg_update, lr_global=1.0)

        if mode == "AS" and flags.adapt_clip:
            C_r = update_clip_bound(C_r, raw_norms, cfg.clip_adapt)

        vm = eval_metrics(global_model, Xva, yva, batch_size=cfg.batch_size)
        hist["round"].append(r)
        hist["val_auprc"].append(vm["auprc"])
        hist["clip_bound"].append(float(C_r if (mode == "AS" and flags.adapt_clip) else (cfg.fixed_clip_norm if mode != "NO_DP" else 0.0)))
        hist["epsL_r"].append(float(epsL_r))
        hist["epsC_r"].append(float(epsC_r))
        hist["lap_b_med"].append(float(np.median(client_sens_list) / max(epsL_r, 1e-12) if client_sens_list and epsL_r > 0 else 0.0))
        hist["sigma"].append(float(sigma))

        if vm["auprc"] > best_val + 1e-4:
            best_val = vm["auprc"]
            best_state = copy.deepcopy(global_model.state_dict())
        prev_val = vm["auprc"]

    if best_state is not None:
        global_model.load_state_dict(best_state)
    test_metrics = eval_metrics(global_model, Xte, yte, batch_size=cfg.batch_size)
    privacy = {
        "epsL_total_target": float(cfg.eps_ldp_total),
        "epsC_total_target": float(cfg.eps_cdp_total),
        "epsL_total_used": float(epsL_used),
        "epsC_total_used": float(epsC_used),
        "eps_hybrid_total_used": float(epsL_used + epsC_used),
        "delta_hybrid_total": float(cfg.delta_cdp if mode != "NO_DP" else 0.0)
    }
    return global_model, pd.DataFrame(hist), test_metrics, privacy

def make_plots(hist_df, out_dir):
    paths = []
    for col, title in [("val_auprc", "Validation AUPRC"), ("clip_bound", "Adaptive Clip Bound"), ("epsL_r", "Local DP Budget per Round"), ("epsC_r", "Central DP Budget per Round")]:
        plt.figure()
        plt.plot(hist_df["round"], hist_df[col])
        plt.xlabel("Round")
        plt.ylabel(col)
        plt.title(title)
        p = os.path.join(out_dir, f"{col}.png")
        plt.savefig(p, bbox_inches="tight", dpi=160)
        plt.close()
        paths.append(p)
    return paths

def run_platform_training(dataset_file, target_column, positive_label, privacy_mode,
                          n_clients, rounds, local_epochs, batch_size, learning_rate,
                          eps_ldp_total, eps_cdp_total, delta_cdp, fixed_clip_norm,
                          seed, use_pos_weight):
    if dataset_file is None:
        raise gr.Error("Please upload a CSV dataset.")
    out_dir = tempfile.mkdtemp(prefix="pr_as_hdp_outputs_")
    Xtr, Xva, Xte, ytr, yva, yte, scaler, meta, feature_names = load_uploaded_csv(
        dataset_file, target_column, positive_label, seed=int(seed)
    )
    mode_map = {
        "No DP baseline": "NO_DP",
        "Fixed Hybrid DP": "FIXED_HDP",
        "A4 Full AS-HDP++": "AS"
    }
    mode = mode_map[privacy_mode]
    flags = None if mode != "AS" else AblationFlags(True, True, True, True)
    cfg = FLConfig(
        n_clients=int(n_clients), rounds=int(rounds), local_epochs=int(local_epochs),
        batch_size=int(batch_size), lr=float(learning_rate),
        fixed_clip_norm=float(fixed_clip_norm), eps_ldp_total=float(eps_ldp_total),
        eps_cdp_total=float(eps_cdp_total), delta_cdp=float(delta_cdp),
        use_pos_weight=bool(use_pos_weight)
    )
    model, hist, metrics, privacy = run_fl_mode(mode, Xtr, ytr, Xva, yva, Xte, yte, Xtr.shape[1], cfg, flags, int(seed))

    prob = predict_proba(model, Xte, cfg.batch_size)
    pred = (prob >= 0.5).astype(int)
    pred_df = pd.DataFrame({"y_true": yte, "predicted_probability": prob, "predicted_class": pred})
    metrics_df = pd.DataFrame([metrics])
    privacy_df = pd.DataFrame([privacy])
    meta_df = pd.DataFrame([meta])

    pred_path = os.path.join(out_dir, "predictions.csv")
    hist_path = os.path.join(out_dir, "training_history.csv")
    metrics_path = os.path.join(out_dir, "metrics.csv")
    privacy_path = os.path.join(out_dir, "privacy_ledger.csv")
    model_path = os.path.join(out_dir, "trained_model.pt")
    pred_df.to_csv(pred_path, index=False)
    hist.to_csv(hist_path, index=False)
    metrics_df.to_csv(metrics_path, index=False)
    privacy_df.to_csv(privacy_path, index=False)
    torch.save(model.state_dict(), model_path)

    plot_paths = make_plots(hist, out_dir)
    zip_path = os.path.join(out_dir, "PR_AS_HDP_outputs.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in [pred_path, hist_path, metrics_path, privacy_path, model_path] + plot_paths:
            z.write(p, arcname=os.path.basename(p))

    summary = f"""
### Training completed

**Dataset:** {meta['rows']} rows, {meta['features']} features
**Target column:** {meta['target']}
**Positive class rate:** {meta['positive_rate']:.3f}
**Mode:** {privacy_mode}
**Clients:** {n_clients}
**Rounds:** {rounds}

**Test Accuracy:** {metrics['accuracy']:.4f}
**Test Precision:** {metrics['precision']:.4f}
**Test Recall:** {metrics['recall']:.4f}
**Test F1:** {metrics['f1']:.4f}
**Test AUPRC:** {metrics['auprc']:.4f}
**Test AUROC:** {metrics.get('auroc', np.nan):.4f}

**Hybrid privacy budget used:** ε = {privacy['eps_hybrid_total_used']:.4f}, δ = {privacy['delta_hybrid_total']:.1e}
"""
    return summary, metrics_df, privacy_df, hist, pred_df.head(50), plot_paths[0], plot_paths[1], zip_path



import gradio as gr

with gr.Blocks(title="FL-AS-HDP Healthcare IoT Platform", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # FL-AS-HDP Healthcare IoT Platform
    Upload a tabular healthcare dataset, configure the federated learning and privacy settings, run the model, and download predictions.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            dataset_file = gr.File(label="Upload CSV dataset", file_types=[".csv"])
            target_column = gr.Textbox(label="Target column name (optional)", placeholder="Example: Outcome, diagnosis, Grade")
            positive_label = gr.Textbox(label="Positive class label (optional)", placeholder="Example: M, HGG, 1")

            privacy_mode = gr.Radio(
                ["No DP baseline", "Fixed Hybrid DP", "A4 Full AS-HDP++"],
                value="A4 Full AS-HDP++",
                label="Privacy mode"
            )

            with gr.Accordion("Federated learning settings", open=True):
                n_clients = gr.Slider(2, 20, value=8, step=1, label="Number of clients")
                rounds = gr.Slider(1, 100, value=30, step=1, label="Communication rounds")
                local_epochs = gr.Slider(1, 10, value=3, step=1, label="Local epochs")
                batch_size = gr.Dropdown([32, 64, 128, 256, 512, 1024], value=256, label="Batch size")
                learning_rate = gr.Number(value=9e-4, label="Learning rate")
                use_pos_weight = gr.Checkbox(value=True, label="Use positive class weight for imbalanced data")

            with gr.Accordion("Privacy settings", open=True):
                eps_ldp_total = gr.Number(value=12.0, label="Total local DP epsilon")
                eps_cdp_total = gr.Number(value=8.0, label="Total central DP epsilon")
                delta_cdp = gr.Number(value=1e-5, label="Central DP delta")
                fixed_clip_norm = gr.Number(value=1.0, label="Fixed clipping norm")
                seed = gr.Number(value=42, label="Random seed", precision=0)

            run_btn = gr.Button("Run Federated Model", variant="primary")

        with gr.Column(scale=2):
            summary = gr.Markdown()
            with gr.Tab("Metrics"):
                metrics_table = gr.Dataframe(label="Test metrics")
                privacy_table = gr.Dataframe(label="Privacy ledger")
            with gr.Tab("Training history"):
                history_table = gr.Dataframe(label="Round-by-round history")
                with gr.Row():
                    auprc_plot = gr.Image(label="Validation AUPRC")
                    clip_plot = gr.Image(label="Clip bound")
            with gr.Tab("Predictions"):
                pred_table = gr.Dataframe(label="First 50 test predictions")
            output_zip = gr.File(label="Download all outputs")

    run_btn.click(
        fn=run_platform_training,
        inputs=[
            dataset_file, target_column, positive_label, privacy_mode,
            n_clients, rounds, local_epochs, batch_size, learning_rate,
            eps_ldp_total, eps_cdp_total, delta_cdp, fixed_clip_norm,
            seed, use_pos_weight
        ],
        outputs=[
            summary, metrics_table, privacy_table, history_table,
            pred_table, auprc_plot, clip_plot, output_zip
        ]
    )

demo.launch(share=True, debug=False)


Device: cpu


/tmp/ipykernel_7920/1901129936.py:596: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="FL-AS-HDP Healthcare IoT Platform", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a122d6505aa8c08e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
